# 03 — Model-specific validation

Validation diagnostics that depend on **the chosen SCM model** (the fit's gap series, weights, RMSPE). For each of the 5 ensemble models from 02 × each event, we report:

| § | Diagnostic | What it tests |
|---|---|---|
| **§5a (i)** | Walk-forward hold-out CV | Does the model generalize from train (80%) to val (20%) within pre-event? |
| **§5a (ii)** | Moment matching | Pre-period mean, SD, min, max, AR(1) of log-Brent vs synthetic |
| **§5b** | Parallel-fit defence | Statistics on the pre-period gap series (mean, AR(1), trend slope, R²) |

**No pass/fail thresholds** are applied here, following the report-and-interpret convention of the SCM literature (Abadie 2010/2015/2021) and the Roth (2022) critique of pretests with low statistical power. The diagnostics inform interpretation (drift correction, model-comparison ranking, narrative for the discussion) rather than gatekeeping ensemble inclusion.

## Out of scope here — see [01.5_Donor_Cleanliness.ipynb](01.5_Donor_Cleanliness.ipynb)

Model-**agnostic** tests on the donor pool itself (per-donor SUTVA cleanliness + within-pre-period regime stability) live in 01.5 because they don't depend on the model choice. The model-**native** in-space placebo (§5e (i) in validation.md) lives in `04_Inference.ipynb`.

**Inputs**: saved fits from `data/results/{event}/preferred/shared/{model}/fit.pkl` (produced by `02_Fit_Models.ipynb`).
**Outputs**: validation tables in `data/validation/*.csv`.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from lib.config import T0, PRE_WINDOWS, MODEL_HPARAMS, DONOR_POOL_VARIANT
from lib.data import build_panel, load_fit, list_fits, save_validation_table
from lib.validation import (walk_forward_cv, parallel_fit_test, moment_matching,
                            regime_stability_test, event_window_return_test,
                            chow_break_test_bootstrap, benjamini_hochberg)

MODELS_AVAILABLE = ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bsts']
EVENTS = ['russia', 'hormuz']
WINDOW = 'preferred'   # validate the preferred specification only
VARIANT = DONOR_POOL_VARIANT

print(f'Validating event = {EVENTS}, window = {WINDOW}, variant = {VARIANT}')
print(f'Saved fits available: {len(list_fits())}')

Validating event = ['russia', 'hormuz'], window = preferred, variant = shared
Saved fits available: 30
Last run: 2026-05-25 20:36:40


## §5a (i) — Walk-forward hold-out CV

Fits each model on the first 80% of the pre-period and evaluates on the last 20%. A model whose validation RMSE is much larger than its train RMSE is overfitting and gets a `passes=False` flag.

In [2]:
from lib.validation import get_tuned_hparams

wf_rows = []
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    for model in MODELS_AVAILABLE:
        # Use the val-tuned hyperparameters that 02_Fit_Models stored with each fit,
        # falling back to config defaults for any keys not tuned (e.g. n_random_v for SCM).
        kwargs = {**MODEL_HPARAMS.get(model, {}),
                  **get_tuned_hparams(model, event, WINDOW, VARIANT)}
        try:
            r = walk_forward_cv(model, panel, 'Brent', meta['donors'],
                                t0=meta['t0'], t_pre_start=meta['t_pre_start'], **kwargs)
            r.update({'event': event, 'window': WINDOW})
            wf_rows.append(r)
        except Exception as e:
            wf_rows.append({'event': event, 'window': WINDOW, 'model': model,
                            'error': str(e)[:60]})

wf_df = pd.DataFrame(wf_rows)
save_validation_table(wf_df, 'walk_forward_cv')
wf_df.round(4)

,model,train_rmse,val_rmse,val_train_ratio,n_train,n_val,passes,event,window
0,convex_scm,0.1115,0.0858,0.7701,336,85,True,russia,preferred
1,ascm,0.0457,0.0872,1.9094,336,85,True,russia,preferred
2,elastic_net,0.0553,0.0939,1.6987,336,85,True,russia,preferred
3,xgboost,0.0380,0.1323,3.4805,336,85,False,russia,preferred
4,bsts,0.0335,0.1710,5.0991,336,85,False,russia,preferred
5,convex_scm,0.0589,0.1287,2.1848,338,85,False,hormuz,preferred
6,ascm,0.0349,0.0872,2.4963,338,85,False,hormuz,preferred
7,elastic_net,0.0431,0.0487,1.1307,338,85,True,hormuz,preferred
8,xgboost,0.0451,0.0991,2.1958,338,85,False,hormuz,preferred
9,bsts,0.0325,0.0934,2.8728,338,85,False,hormuz,preferred


Last run: 2026-05-25 20:36:46


## §5b — Pre-period parallel-fit defence

Diagnostics on each model's pre-period gap series. **Reported and interpreted, not pass/fail tested.**

The `passes` column from `parallel_fit_test()` retains the historical pass/fail logic for backward compatibility but should not be read as a gatekeeper — per validation.md §5b, the SCM literature (Abadie 2010/2015/2021) does not impose formal pre-trend thresholds, and the Roth (2022) "Pretest with caution" critique shows that formal pre-trend tests have low power at typical applied sample sizes.

**How to read the numbers:** large `slope_pct_per_year` magnitudes signal pre-period drift. Use the slope to drift-adjust the post-event gap if needed: `drift_pct = slope × (post-window in years)`. For Russia post-window ≈ 0.58 yr; Hormuz ≈ 0.25 yr.

In [3]:
pf_rows = []
for event in EVENTS:
    for model in MODELS_AVAILABLE:
        fit = load_fit(event, WINDOW, model, variant=VARIANT)
        if fit is None:
            continue
        r = parallel_fit_test(fit)
        r.update({'event': event, 'window': WINDOW, 'model': model})
        pf_rows.append(r)

pf_df = pd.DataFrame(pf_rows)
save_validation_table(pf_df, 'parallel_fit_defence')
pf_df.round(4)

,n,mean_pct,sd_pct,t_stat,p_mean_zero,ar1,slope_pct_per_year,p_slope_zero,r_squared,passes,event,window,model
0,421,0.4285,10.4786,0.8392,0.4019,0.9795,10.0104,0.0000,0.2097,False,russia,preferred,convex_scm
1,421,0.1140,4.8075,0.4866,0.6268,0.8811,0.3351,0.4942,0.0011,True,russia,preferred,ascm
2,421,0.1738,5.9482,0.5996,0.5491,0.9167,1.1874,0.0498,0.0092,False,russia,preferred,elastic_net
3,421,0.0799,3.9598,0.4139,0.6792,0.8571,3.5064,0.0000,0.1801,False,russia,preferred,xgboost
4,421,0.0932,4.3252,0.4422,0.6586,0.8517,0.1519,0.7305,0.0003,True,russia,preferred,bsts
5,423,0.2611,6.7584,0.7945,0.4273,0.9589,-7.0425,0.0000,0.2535,False,hormuz,preferred,convex_scm
6,423,0.0668,3.6638,0.3748,0.7080,0.8225,-0.1337,0.7178,0.0003,True,hormuz,preferred,ascm
7,423,0.0995,4.4997,0.4546,0.6496,0.9010,-0.5833,0.1986,0.0039,False,hormuz,preferred,elastic_net
8,423,0.1446,4.8122,0.6182,0.5368,0.9223,-4.6837,0.0000,0.2211,False,hormuz,preferred,xgboost
9,423,0.0603,3.4840,0.3558,0.7221,0.7744,-0.0534,0.8794,0.0001,True,hormuz,preferred,bsts


Last run: 2026-05-25 20:36:46


## §5a (ii) — Moment matching

Pre-period mean, SD, min, max, AR(1) of log-Brent vs log-synthetic. Δ Mean ≈ 0 is required; |Δ SD| > 25% of treated SD indicates the donor pool cannot span Brent's volatility.

In [4]:
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    for model in MODELS_AVAILABLE:
        fit = load_fit(event, WINDOW, model, variant=VARIANT)
        if fit is None:
            continue
        df = moment_matching(fit, panel, 'Brent')
        df['event'] = event
        df['model'] = model
        save_validation_table(df, f'moments_{event}_{model}')
        print(f'\n{event} / {model}:')
        print(df.round(4).to_string())


russia / convex_scm:
      treated   synth   delta  delta_pct   event       model
mean   4.1281  4.1294  0.0013     0.0315  russia  convex_scm
sd     0.2693  0.2071 -0.0622   -23.0891  russia  convex_scm
min    3.5926  3.7625  0.1699     4.7287  russia  convex_scm
max    4.6216  4.4769 -0.1448    -3.1322  russia  convex_scm
ar1    0.9969  0.9983  0.0014     0.1383  russia  convex_scm

russia / ascm:
      treated   synth   delta  delta_pct   event model
mean   4.1281  4.1281  0.0000     0.0001  russia  ascm
sd     0.2693  0.2634 -0.0059    -2.1930  russia  ascm
min    3.5926  3.6817  0.0890     2.4780  russia  ascm
max    4.6216  4.5642 -0.0574    -1.2419  russia  ascm
ar1    0.9969  0.9971  0.0003     0.0262  russia  ascm

russia / elastic_net:
      treated   synth   delta  delta_pct   event        model
mean   4.1281  4.1281 -0.0000    -0.0000  russia  elastic_net
sd     0.2693  0.2573 -0.0120    -4.4499  russia  elastic_net
min    3.5926  3.6393  0.0466     1.2983  russia  elastic

## Validation summary — numerical only

Per (event, model): walk-forward train/val RMSE + parallel-fit mean and slope + implied drift contribution. **No pass/fail flags** — interpret the numbers narratively (cf. validation.md §5a and §5b). The `wf_passes` / `pf_passes` columns from the legacy strict-threshold rule are retained in the upstream CSV files but should not be read as exclusion criteria.

In [5]:
summary_rows = []
for event in EVENTS:
    post_yr = {'russia': 0.58, 'hormuz': 0.25}.get(event, np.nan)
    for model in MODELS_AVAILABLE:
        wf = wf_df[(wf_df.get('event') == event) & (wf_df.get('model') == model)]
        pf = pf_df[(pf_df.get('event') == event) & (pf_df.get('model') == model)]
        slope = pf['slope_pct_per_year'].iloc[0] if len(pf) else np.nan
        drift_pct = slope * post_yr if not np.isnan(slope) else np.nan
        summary_rows.append({
            'event': event, 'model': model,
            'wf_train_rmse': wf['train_rmse'].iloc[0] if len(wf) else np.nan,
            'wf_val_rmse':   wf['val_rmse'].iloc[0]   if len(wf) else np.nan,
            'wf_ratio':      wf['val_train_ratio'].iloc[0] if len(wf) else np.nan,
            'pf_mean_pct':   pf['mean_pct'].iloc[0] if len(pf) else np.nan,
            'pf_slope_yr':   slope,
            'pf_r2':         pf['r_squared'].iloc[0] if len(pf) else np.nan,
            'drift_contribution_pct': drift_pct,
        })
summary_df = pd.DataFrame(summary_rows)
save_validation_table(summary_df, 'validation_summary')
print('Per (event, model) validation diagnostics:')
print('  wf_*: out-of-sample (walk-forward, model fit on train only)')
print('  pf_*: in-sample pre-period gap series statistics from the saved fit')
print('  drift_contribution_pct = pre-period slope × post-window length (interpretive only)')
print()
print(summary_df.round(4).to_string(index=False))

Per (event, model) validation diagnostics:
  wf_*: out-of-sample (walk-forward, model fit on train only)
  pf_*: in-sample pre-period gap series statistics from the saved fit
  drift_contribution_pct = pre-period slope × post-window length (interpretive only)

 event       model  wf_train_rmse  wf_val_rmse  wf_ratio  pf_mean_pct  pf_slope_yr  pf_r2  drift_contribution_pct
russia  convex_scm         0.1115       0.0858    0.7701       0.4285      10.0104 0.2097                  5.8060
russia        ascm         0.0457       0.0872    1.9094       0.1140       0.3351 0.0011                  0.1943
russia elastic_net         0.0553       0.0939    1.6987       0.1738       1.1874 0.0092                  0.6887
russia     xgboost         0.0380       0.1323    3.4805       0.0799       3.5064 0.1801                  2.0337
russia        bsts         0.0335       0.1710    5.0991       0.0932       0.1519 0.0003                  0.0881
hormuz  convex_scm         0.0589       0.1287    2.184